# 数据导入和预处理

### 读入文件 (特征提取已经手动完成)

In [1]:
import pandas as pd
# 这里假设数据集里有一个主要的 CSV 文件，实际要根据 path 下的文件情况修改文件名
df = pd.read_csv('preprepared.csv', encoding='latin-1')
df=df[df['label']!=0]

####  用均值代替缺失值，调整标签

In [2]:
for col in df.columns:
    # 计算该列非0值的平均值
    col_mean = df[df[col] != 0][col].mean()
    # 将该列中值为0的单元格替换为计算出的平均值
    df[col] = df[col].replace(0, col_mean)
df['label']=df["label"].replace(2,0)

# 模型代码

### 模型函数的书写

In [ ]:
import torch.nn as nn
import torch
device=torch.device('xpu')
class DropoutClassifier(nn.Module):
    def __init__(self, input_size):
        super(DropoutClassifier,self).__init__()
        self.l1=nn.Linear(18,32)
        self.relu=nn.ReLU()
        self.l2=nn.Linear(32,2)
    def forward(self,x):
        # TODO:
        x=self.relu(self.l1(x))
        x=self.l2(x)
        return x

### 基础设置

#### 函数定义

In [4]:
model= DropoutClassifier(64).to(device)
critrerion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=0.001,weight_decay=0.0001)
epoches =20

#### 数据定义


In [5]:
from sklearn.model_selection import train_test_split as st
from torch.utils.data import DataLoader, TensorDataset

df_train, df_test = st(df, test_size=0.2)

x_train = df_train.drop("label", axis=1)
y_train = df_train["label"]
x_train = torch.tensor(x_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

x_test = df_test.drop("label", axis=1)
y_test = df_test["label"]
x_test = torch.tensor(x_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.long)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) 

#### 训练

In [6]:
for epoch in range(epoches):
    model.train()
    for batch in train_loader :
        inputs,label=batch
        inputs=inputs.to(device)
        label=label.to(device)
        optimizer.zero_grad()
        output=model(inputs)
        loss=critrerion(output,label)
        loss.backward()
        optimizer.step()
        

#### 验证训练准确率

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score

model.eval()
running_loss =0.0
right=0
with torch.no_grad():
    for batch in test_loader:
        inputs,label=batch
        inputs=inputs.to(device)
        label=label.to(device)
        optimizer.zero_grad()
        output=model(inputs)
        sxp=np.argmax(output.cpu().numpy(),axis=1)
        a=accuracy_score(sxp, label, normalize=True, sample_weight=None)
        loss=critrerion(output,label)
        running_loss +=loss
        right+=a*64
print(running_loss)
print("accuracy",right/3995)

tensor(5.6706)
accuracy 0.9893941500950262
